Retrieval quality depends heavily on chunk size selection.
Smaller chunks improve precision but sometimes lose full context.
Larger chunks preserve context but may introduce irrelevant information.
Multilingual documents require multilingual embedding models and language detection.
Metadata filtering helps improve relevance by narrowing document selection.
Re-ranking improves final results by combining semantic similarity and keyword matching.
Hybrid search increases accuracy by combining dense and sparse retrieval methods.
Efficient knowledge base updates require incremental embedding generation.
Monitoring precision, recall, and user feedback is important in production systems.
Optimizing computational cost involves caching, batching, and using vector databases for scalability.

In [ ]:
# # Lab: RAG Basics - Building Retrieval Pipeline

# ## Learning Objectives

# By the end of this lab, you will be able to:
# - Understand the core components of a retrieval pipeline
# - Implement document chunking strategies
# - Create and work with text embeddings
# - Build a vector-based retrieval system
# - Evaluate retrieval quality

# ## Introduction

# Retrieval-Augmented Generation (RAG) is a technique that enhances language models by retrieving relevant information from a knowledge base before generating responses. The retrieval pipeline is the foundation of any RAG system, responsible for finding the most relevant information given a query.

# In this lab, you will build a complete retrieval pipeline from scratch, learning each component step by step.


In [ ]:


## Setup

### Required Libraries

# Install the following libraries:
# ```bash
!pip install sentence-transformers numpy scikit-learn

### Sample Dataset

# For this lab, we'll work with a collection of documents about artificial intelligence concepts. Create a file called `documents.py` with the following content:

# ```python
DOCUMENTS= [
    "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
    "Deep learning is a type of machine learning based on artificial neural networks with multiple layers, allowing computers to learn complex patterns.",
    "Natural language processing (NLP) is a field of AI that focuses on the interaction between computers and human language, enabling machines to understand and generate text.",
    "Computer vision is an AI field that trains computers to interpret and understand visual information from the world, such as images and videos.",
    "Reinforcement learning is a machine learning approach where an agent learns to make decisions by performing actions and receiving rewards or penalties.",
    "Transfer learning is a technique where a model developed for one task is reused as the starting point for a model on a second task.",
    "Neural networks are computing systems inspired by biological neural networks, consisting of interconnected nodes that process information.",
    "Supervised learning involves training a model on labeled data, where the correct output is known for each input example.",
    "Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.",
    "Transformers are a type of neural network architecture that uses self-attention mechanisms, revolutionary for NLP tasks."
]


In [ ]:

### Sample Dataset

# For this lab, we'll work with a collection of documents about artificial intelligence concepts. Create a file called `documents.py` with the following content:

# ```python
DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
    "Deep learning is a type of machine learning based on artificial neural networks with multiple layers, allowing computers to learn complex patterns.",
    "Natural language processing (NLP) is a field of AI that focuses on the interaction between computers and human language, enabling machines to understand and generate text.",
    "Computer vision is an AI field that trains computers to interpret and understand visual information from the world, such as images and videos.",
    "Reinforcement learning is a machine learning approach where an agent learns to make decisions by performing actions and receiving rewards or penalties.",
    "Transfer learning is a technique where a model developed for one task is reused as the starting point for a model on a second task.",
    "Neural networks are computing systems inspired by biological neural networks, consisting of interconnected nodes that process information.",
    "Supervised learning involves training a model on labeled data, where the correct output is known for each input example.",
    "Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.",
    "Transformers are a type of neural network architecture that uses self-attention mechanisms, revolutionary for NLP tasks."
]

In [ ]:

## Part 1: Document Preprocessing and Chunking

### Understanding the Importance

# Before we can retrieve documents, we need to prepare them properly. In real-world scenarios, documents are often too large to process as single units. Chunking breaks documents into smaller, manageable pieces while preserving context.

# **Why this matters:** Poor chunking can lead to lost context or irrelevant retrievals, directly impacting the quality of your RAG system.

# ### Task 1.1: Implement Basic Chunking

# Create a function that splits documents into chunks based on a maximum token/word count.


def chunk_by_words(text, max_words=50, overlap=10):
    """
    Split text into chunks based on word count.

    Args:
        text: Input text to chunk
        max_words: Maximum words per chunk
        overlap: Number of overlapping words between chunks

    Returns:
        List of text chunks
    """
    # Your code here
        # Validate input
    if not isinstance(text, str):
        raise ValueError("Input text must be a string.")

    if max_words <= 0:
        raise ValueError("max_words must be greater than 0.")

    if overlap >= max_words:
        raise ValueError("overlap must be smaller than max_words.")

    # Split text into individual words
    words = text.split()

    chunks = []
    start = 0

    # Step size determines how much we move forward each time
    step = max_words - overlap

    # Create chunks using sliding window approach
    while start < len(words):
        end = start + max_words
        chunk_words = words[start:end]
        chunk = " ".join(chunk_words)
        chunks.append(chunk)

        start += step  # Move window forward

    return chunks

# **Test your implementation:**

# ```python
sample_text = "Machine learning is a subset of artificial intelligence. It enables systems to learn from experience. Deep learning uses neural networks with multiple layers."

chunks = chunk_by_words(sample_text, max_words=10, overlap=3)
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
      print(f"Chunk {i+1}: {chunk}")


Number of chunks: 4
Chunk 1: Machine learning is a subset of artificial intelligence. It enables
Chunk 2: intelligence. It enables systems to learn from experience. Deep learning
Chunk 3: experience. Deep learning uses neural networks with multiple layers.
Chunk 4: multiple layers.


In [ ]:


### Task 1.2: Sentence-Based Chunking

#Implement a more sophisticated chunking strategy that respects sentence boundaries.


### Task 1.2: Sentence-Based Chunking

# Implement a more sophisticated chunking strategy that respects sentence boundaries.
# Sentence-based chunking ensures that each chunk preserves complete thoughts.
# This improves semantic understanding during embedding and retrieval.


def chunk_by_sentences(text, max_sentences=2):
    """
    Split text into chunks based on complete sentences.

    Args:
        text: Input text to chunk
        max_sentences: Maximum sentences per chunk

    Returns:
        List of text chunks
    """

    # Validate input to improve robustness
    if not isinstance(text, str):
        raise ValueError("Input text must be a string.")

    if max_sentences <= 0:
        raise ValueError("max_sentences must be greater than 0.")

    # First split text into sentences using ". "
    # This is a simple approach and works well for clean text.
    # In real-world systems, more advanced NLP sentence tokenizers can be used.
    sentences = text.split(". ")

    chunks = []
    temp = []

    # Loop through sentences
    # We use a temporary list to group sentences before forming a chunk.
    for s in sentences:
        temp.append(s.strip())

        # If reached max_sentences, make one chunk
        # This ensures each chunk contains exactly max_sentences sentences (except last).
        if len(temp) == max_sentences:
            chunk = ". ".join(temp)

            # Add period if not present
            # This ensures sentence structure remains grammatically complete.
            if not chunk.endswith("."):
                chunk += "."

            chunks.append(chunk)
            temp = []

    # If some sentences are left
    # This handles the remaining sentences that didn't reach max_sentences count.
    if len(temp) > 0:
        chunk = ". ".join(temp)
        if not chunk.endswith("."):
            chunk += "."
        chunks.append(chunk)

    # Returning final list of sentence-based chunks
    return chunks

    # Your code here
    # Hint: Use simple sentence splitting on periods followed by spaces
    pass


# 1) What are the advantages of sentence-based chunking over word-based chunking?
print("Advantages of sentence-based chunking:")
print("It keeps full sentence meaning.")
print("It does not break sentence in middle.")
print("Embedding model understands full sentence better.")
print("Retrieval results are more accurate.")

# 2) When might word-based chunking be more appropriate?
print("\nWhen word-based chunking is better:")
print("If text is very long.")
print("If we want fixed size chunks.")
print("If sentences are too large.")
print("When memory limit is small.")

Advantages of sentence-based chunking:
It keeps full sentence meaning.
It does not break sentence in middle.
Embedding model understands full sentence better.
Retrieval results are more accurate.

When word-based chunking is better:
If text is very long.
If we want fixed size chunks.
If sentences are too large.
When memory limit is small.


In [ ]:
The secret `HF_TOKEN` does not exist in your Colab secrets.

It is only a warning, not an error.
Authentication is optional for public models like:

all-MiniLM-L6-v2

In [2]:
## Part 2: Creating Embeddings

### Understanding the Importance

# Embeddings convert text into numerical vectors that capture semantic meaning. This transformation is crucial because computers can't directly compare text semantically, but they can measure distances between vectors.

# **Why this matters:** The quality of embeddings directly determines how well your retrieval system understands semantic similarity. Poor embeddings will retrieve irrelevant documents even if your retrieval logic is perfect.

### Task 2.1: Initialize Embedding Model

# Load a pre-trained sentence transformer model.

from sentence_transformers import SentenceTransformer

def initialize_embedding_model(model_name='all-MiniLM-L6-v2'):
    """
    Initialize a sentence transformer model for creating embeddings.

    Args:
        model_name: Name of the pre-trained model

    Returns:
        Loaded model
    """

    # Validate input to ensure model_name is provided correctly
    if not isinstance(model_name, str) or len(model_name) == 0:
        raise ValueError("model_name must be a non-empty string.")

    # Load the model using given name
    # 'all-MiniLM-L6-v2' is lightweight, fast, and performs well for semantic similarity.
    # It balances performance and computational efficiency.
    try:
        model = SentenceTransformer(model_name)
        return model
    except Exception as e:
        print("Error loading model:", e)
        return None

    # Return the loaded model
    # Returning the model allows reuse for multiple embedding operations.
    return model

# This ensures that the model loads correctly without runtime errors.
# Testing the function
model = initialize_embedding_model()
print("Model loaded successfully")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:


### Task 2.2: Generate Document Embeddings

# Create embeddings for all documents in your knowledge base.

# ```python
def create_embeddings(documents, model):
    """
    Generate embeddings for a list of documents.

    Args:
        documents: List of text documents
        model: Embedding model

    Returns:
        numpy array of embeddings
    """

    # Validate inputs to ensure robustness
    if not isinstance(documents, list):
        raise ValueError("documents must be a list of text strings.")

    if len(documents) == 0:
        raise ValueError("documents list cannot be empty.")

    # convert documents into embeddings
    # model.encode() converts each document into a high-dimensional vector.
    # Each vector captures semantic meaning of the corresponding document.
    # Similar documents will have vectors that are closer in vector space.
    embeddings = model.encode(documents)

    # Returning embeddings as a numpy array.
    # Shape will be (number_of_documents, embedding_dimension)
    return embeddings

# **Test your implementation:**

# ```python


# ```python

# Initialize model for embedding generation
# This ensures we are using the same embedding space for all documents.
model = initialize_embedding_model()

# Generate embeddings for all documents
# Each document will now have a fixed-length vector representation.
embeddings = create_embeddings(DOCUMENTS, model)

# Print number of documents to verify alignment
print(f"Number of documents: {len(DOCUMENTS)}")

# Print embedding shape to confirm dimensions
# Expected shape: (number_of_documents, embedding_dimension)
print(f"Embedding shape: {embeddings.shape}")

# Display first 10 dimensions of first embedding for inspection
# This helps verify that embeddings are numerical vectors.
print(f"First embedding (first 10 dimensions): {embeddings[0][:10]}")
# ```
# ```

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of documents: 10
Embedding shape: (10, 384)
First embedding (first 10 dimensions): [ 0.00840915 -0.0036058   0.05522151  0.06061025  0.03103591 -0.02438609
 -0.02974015 -0.0305842  -0.0707221  -0.00281702]


In [ ]:
### Task 2.3: Understanding Embedding Space

# Calculate and visualize the similarity between different documents.
# Cosine similarity is commonly used in embedding space to measure semantic closeness.
# It measures the angle between two vectors, not their magnitude.

# ```python
import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.

    Args:
        vec1: First vector
        vec2: Second vector

    Returns:
        Similarity score between -1 and 1
    """

    # Validate inputs for robustness
    if len(vec1) != len(vec2):
        raise ValueError("Vectors must have the same dimensions.")

    # Dot product measures directional similarity
    dot_product = np.dot(vec1, vec2)

    # Compute magnitudes (L2 norms) of both vectors
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    # Prevent division by zero
    if norm_vec1 == 0 or norm_vec2 == 0:
        raise ValueError("One of the vectors has zero magnitude.")

    # Cosine similarity formula
    # similarity = (A · B) / (||A|| * ||B||)
    similarity = dot_product / (norm_vec1 * norm_vec2)

    return similarity
# ```

# **Experiment:**

# Compare similarities between different document pairs
# Document 0: Machine learning definition
# Document 1: Deep learning definition
# Document 4: Reinforcement learning definition

similarity_ml_dl = cosine_similarity(embeddings[0], embeddings[1])
similarity_ml_rl = cosine_similarity(embeddings[0], embeddings[4])

print(f"Similarity (ML vs DL): {similarity_ml_dl:.4f}")
print(f"Similarity (ML vs RL): {similarity_ml_rl:.4f}")


# **Questions:**
print("\nWhich pair is more similar?")
print("Machine Learning and Deep Learning are more similar because Deep Learning is a part of Machine Learning.")

print("\nWhat does the similarity score tell?")
print("The similarity score shows how close two documents are in meaning.")
print("Value close to 1 means very similar.")
print("Value close to 0 means less related.")
print("Value close to -1 means opposite meaning.")


Similarity (ML vs DL): 0.5919
Similarity (ML vs RL): 0.6325

Which pair is more similar?
Machine Learning and Deep Learning are more similar because Deep Learning is a part of Machine Learning.

What does the similarity score tell?
The similarity score shows how close two documents are in meaning.
Value close to 1 means very similar.
Value close to 0 means less related.
Value close to -1 means opposite meaning.


In [ ]:

## Part 3: Building the Retrieval System

### Understanding the Importance

# The retrieval system is the core of your pipeline. It takes a query and finds the most relevant documents from your knowledge base. The efficiency and accuracy of this component directly impact the performance of your entire RAG system.

# **Why this matters:** A slow or inaccurate retrieval system will create bottlenecks and provide poor context to your language model, resulting in irrelevant or incorrect responses.

# ### Task 3.1: Implement Basic Retrieval

# Create a function that retrieves the top-k most similar documents for a given query.





def retrieve_documents(query, documents, embeddings, model, top_k=3):
    """
    Retrieve the most relevant documents for a query.

    Args:
        query: Search query string
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Number of documents to retrieve

    Returns:
        List of tuples (document, similarity_score)
    """

    # Validate inputs for reliability
    if not isinstance(query, str):
        raise ValueError("Query must be a string.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    if len(documents) != len(embeddings):
        raise ValueError("Number of documents must match number of embeddings.")

    # Your code here
    # Steps:
    # 1. Create embedding for the query
    # 2. Calculate similarity between query and all documents
    # 3. Sort by similarity and return top_k results

    # 1. Create embedding for the query
    # The query must be transformed into the same embedding space
    # as the documents to enable meaningful similarity comparison.
    query_embedding = model.encode([query])[0]

    similarities = []

    # 2. Calculate similarity between query and all documents
    # We compare the query embedding with each document embedding
    # using cosine similarity.
    for i, doc_embedding in enumerate(embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)

        # Store document along with its similarity score
        similarities.append((documents[i], score))

    # 3. Sort by similarity and return top_k results
    # Higher cosine similarity means more semantically similar.
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Return only the top_k most relevant documents
    return similarities[:top_k]



# Test your retrieval system
query = "How do computers learn from data?"
results = retrieve_documents(query, DOCUMENTS, embeddings, model, top_k=3)

print(f"Query: {query}\n")

for i, (doc, score) in enumerate(results, 1):
    print(f"Result {i} (Score: {score:.4f}):")
    print(f"{doc}\n")

Query: How do computers learn from data?

Result 1 (Score: 0.5245):
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

Result 2 (Score: 0.5062):
Supervised learning involves training a model on labeled data, where the correct output is known for each input example.

Result 3 (Score: 0.4734):
Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.



In [ ]:

### Task 3.2: Implement Retrieval with Threshold

# Enhance your retrieval function to only return documents above a similarity threshold.

# ```python
def retrieve_with_threshold(query, documents, embeddings, model,
                           top_k=3, threshold=0.3):
    """
    Retrieve documents that meet a minimum similarity threshold.

    Args:
        query: Search query string
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Maximum number of documents to retrieve
        threshold: Minimum similarity score

    Returns:
        List of tuples (document, similarity_score)
    """
      # 1. Create embedding for the query
    query_embedding = model.encode([query])[0]

    # 2. Calculate similarity between query and all documents
    similarities = []
    for i, doc_embedding in enumerate(embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)

        # 3. Keep only scores above threshold
        if score >= threshold:
            similarities.append((documents[i], score))
    similarities.sort(key=lambda x: x[1], reverse=True)

    return similarities[:top_k]


# **Experiment with different thresholds:**

# ```python
queries = [
    "What is deep learning?",
    "How do neural networks work?",
    "Tell me about quantum computing"  # Out of domain query
]

for query in queries:
    print(f"\nQuery: {query}")
    results = retrieve_with_threshold(query, DOCUMENTS, embeddings, model,
                                     top_k=3, threshold=0.3)
    print(f"Retrieved {len(results)} documents")
    if results:
        print(f"Top result (Score: {results[0][1]:.4f}): {results[0][0][:100]}...")


print("\nWhat happens when you query about topics not in your knowledge base?")
print("If the topic is not in the knowledge base, similarity scores will be low and no documents may be returned.")

print("\nHow does the threshold help filter irrelevant results?")
print("The threshold removes documents with low similarity scores.")
print("It helps return only relevant documents and avoids unrelated answers.")



Query: What is deep learning?
Retrieved 3 documents
Top result (Score: 0.7955): Deep learning is a type of machine learning based on artificial neural networks with multiple layers...

Query: How do neural networks work?
Retrieved 3 documents
Top result (Score: 0.7190): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...

Query: Tell me about quantum computing
Retrieved 1 documents
Top result (Score: 0.3105): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...

What happens when you query about topics not in your knowledge base?
If the topic is not in the knowledge base, similarity scores will be low and no documents may be returned.

How does the threshold help filter irrelevant results?
The threshold removes documents with low similarity scores.
It helps return only relevant documents and avoids unrelated answers.


In [ ]:

## Part 4: Optimizing the Retrieval Pipeline

### Understanding the Importance

# As your knowledge base grows, retrieval performance becomes critical. You need efficient data structures and algorithms to handle thousands or millions of documents.

# **Why this matters:** A retrieval system that works well with 10 documents might become unusable with 10,000. Learning optimization techniques early prevents scaling problems later.

### Task 4.1: Implement Batch Processing

# Create a function that efficiently handles multiple queries at once.

# ```python
def batch_retrieve(queries, documents, embeddings, model, top_k=3):
    """
    Efficiently retrieve documents for multiple queries.

    Args:
        queries: List of query strings
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Number of documents per query

    Returns:
        Dictionary mapping each query to its results
    """

    # Validate inputs for robustness
    if not isinstance(queries, list):
        raise ValueError("queries must be a list of strings.")
    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")
    if len(documents) != len(embeddings):
        raise ValueError("Number of documents must match number of embeddings.")

    # Your code here
    # Hint: Create embeddings for all queries at once

    # Create embeddings for all queries in a single batch
    # This is more efficient than encoding each query separately.
    query_embeddings = model.encode(queries)

    results_dict = {}

    # Loop through each query embedding
    for idx, query_embedding in enumerate(query_embeddings):
        similarities = []

        # Compute similarity between this query and all documents
        for i, doc_embedding in enumerate(embeddings):
            score = cosine_similarity(query_embedding, doc_embedding)
            similarities.append((documents[i], score))

        # Sort documents by similarity (highest first)
        similarities.sort(key=lambda x: x[1], reverse=True)

        # Store top_k results for the current query
        results_dict[queries[idx]] = similarities[:top_k]

    # Return dictionary mapping each query to its ranked results
    return results_dict


In [ ]:

### Task 4.2: Create a Retrieval Class

# Encapsulate your retrieval pipeline in a reusable class.

# ```python

class RetrievalPipeline:
    """
    A complete retrieval pipeline for RAG systems.
    """

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """Initialize the pipeline with an embedding model."""

        # Initialize embedding model once during object creation.
        # This avoids repeated loading and improves efficiency.
        self.model = initialize_embedding_model(model_name)

        # Store documents and embeddings inside the class state.
        # This encapsulates the knowledge base within the pipeline.
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents):
        """
        Add documents to the knowledge base.

        Args:
            documents: List of text documents
        """

        # Validate input
        if not isinstance(documents, list):
            raise ValueError("documents must be a list of text strings.")

        # Store documents
        self.documents = documents

        # Precompute embeddings once and store them
        # This prevents recomputation during every query.
        self.embeddings = self.model.encode(documents)


    def retrieve(self, query, top_k=3, threshold=0.0):
        """
        Retrieve relevant documents for a query.

        Args:
            query: Search query
            top_k: Number of results
            threshold: Minimum similarity score

        Returns:
            List of (document, score) tuples
        """

        # Ensure documents are added before retrieval
        if self.embeddings is None:
            raise ValueError("No documents added to the pipeline.")

        # Convert query into embedding in same vector space
        query_embedding = self.model.encode([query])[0]

        similarities = []

        # Compare query embedding with all stored document embeddings
        for i, doc_embedding in enumerate(self.embeddings):
            score = cosine_similarity(query_embedding, doc_embedding)

            # Apply similarity threshold filtering
            if score >= threshold:
                similarities.append((self.documents[i], score))

        # Rank documents by similarity
        similarities.sort(key=lambda x: x[1], reverse=True)

        # Return top_k most relevant documents
        return similarities[:top_k]

    def get_stats(self):
        # Provide useful pipeline metadata for monitoring
        return {
            "Total Documents": len(self.documents),
            "Embedding Shape": None if self.embeddings is None else self.embeddings.shape
        }
# **Test your class:**

# ```python


pipeline = RetrievalPipeline()
pipeline.add_documents(DOCUMENTS)

print("Pipeline Statistics:")
print(pipeline.get_stats())

query = "What are neural networks?"
results = pipeline.retrieve(query, top_k=2)

print(f"\nQuery: {query}")
for doc, score in results:
    print(f"Score: {score:.4f} | {doc[:80]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline Statistics:
{'Total Documents': 10, 'Embedding Shape': (10, 384)}

Query: What are neural networks?
Score: 0.7993 | Neural networks are computing systems inspired by biological neural networks, co...
Score: 0.6365 | Deep learning is a type of machine learning based on artificial neural networks ...


In [ ]:

## Part 5: Evaluation and Analysis

### Understanding the Importance

# You can't improve what you don't measure. Evaluation metrics help you understand how well your retrieval system performs and guide optimization efforts.

# **Why this matters:** Without proper evaluation, you won't know if changes to your pipeline improve or degrade performance. In production systems, this can mean the difference between helpful and harmful AI responses.

# ### Task 5.1: Implement Relevance Metrics

# Create functions to measure retrieval quality.

# ```python
def calculate_precision_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate precision at k: proportion of retrieved docs that are relevant.

    Args:
        retrieved_docs: List of retrieved document indices
        relevant_docs: Set of relevant document indices
        k: Number of top results to consider

    Returns:
        Precision score
    """


    # Edge case: if k is larger than retrieved list, adjust safely
    k = min(k, len(retrieved_docs))

    if k == 0:
        return 0

    retrieved_k = retrieved_docs[:k]

    # Convert relevant_docs to set for O(1) lookup if not already
    relevant_docs = set(relevant_docs)

    # Count relevant documents
    relevant_count = sum(1 for doc in retrieved_k if doc in relevant_docs)

    return relevant_count / k



def calculate_recall_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate Recall@K:
    Proportion of total relevant documents that appear in top-k results.

    Formula:
        Recall@K = (Relevant retrieved in top K) / (Total relevant documents)
    """

    # Convert to set for fast lookup
    relevant_docs = set(relevant_docs)

    # Edge case: no relevant documents defined
    if len(relevant_docs) == 0:
        return 0

    k = min(k, len(retrieved_docs))
    retrieved_k = retrieved_docs[:k]

    relevant_count = sum(1 for doc in retrieved_k if doc in relevant_docs)

    return relevant_count / len(relevant_docs)


In [ ]:


### Task 5.2: Create Test Cases

# Define test queries with known relevant documents.

# ```python
# # Test cases: (query, set of relevant document indices)
test_cases = [
    ("What is machine learning?", {0, 7, 8}),  # ML, supervised, unsupervised
    ("Explain neural networks", {1, 6, 9}),     # Deep learning, neural nets, transformers
    ("How does computer vision work?", {3}),    # Computer vision
]

def evaluate_retrieval(pipeline, test_cases, k=3):
    """
    Evaluate retrieval performance on test cases.

    Args:
        pipeline: RetrievalPipeline instance
        test_cases: List of (query, relevant_docs) tuples
        k: Number of results to retrieve

    Returns:
        Dictionary with average precision and recall
    """
    # Your code here
    # For each test case:
    # 1. Retrieve documents
    # 2. Get indices of retrieved documents
    # 3. Calculate precision and recall
    # 4. Average across all test cases
    total_precision = 0
    total_recall = 0

    for query, relevant_docs in test_cases:

        # 1. Retrieve documents
        results = pipeline.retrieve(query, top_k=k)

        # 2. Get indices of retrieved documents
        retrieved_indices = []
        for doc, score in results:
            index = pipeline.documents.index(doc)
            retrieved_indices.append(index)

        # 3. Calculate precision and recall
        precision = calculate_precision_at_k(retrieved_indices, relevant_docs, k)
        recall = calculate_recall_at_k(retrieved_indices, relevant_docs, k)

        total_precision += precision
        total_recall += recall

    # 4. Average across all test cases
    avg_precision = total_precision / len(test_cases)
    avg_recall = total_recall / len(test_cases)

    return {
        "Average Precision": avg_precision,
        "Average Recall": avg_recall
    }

### Task 5.3: Analyze Retrieval Performance

# Run evaluation and analyze the results.
metrics = evaluate_retrieval(pipeline, test_cases, k=3)
## Your code to evaluate and print results

print("Evaluation Results:")
print("Average Precision:", round(metrics["Average Precision"], 4))
print("Average Recall:", round(metrics["Average Recall"], 4))
print("1) Average precision shows how many retrieved documents are relevant.")
print("2) Average recall shows how many relevant documents were successfully retrieved.")

# **Analysis Questions:**
# - What is the average precision and recall of your system?
print("\nAverage precision and recall are shown above.")

# - Which queries perform best/worst? Why?
print("Queries with clear keywords perform better.")
print("If topic is less detailed in knowledge base, performance is lower.")

# - How could you improve retrieval quality?
print("Use better embedding model.")
print("Add more relevant documents.")
print("Tune threshold and top_k values.")

Evaluation Results:
Average Precision: 0.6667
Average Recall: 0.8889
1) Average precision shows how many retrieved documents are relevant.
2) Average recall shows how many relevant documents were successfully retrieved.

Average precision and recall are shown above.
Queries with clear keywords perform better.
If topic is less detailed in knowledge base, performance is lower.
Use better embedding model.
Add more relevant documents.
Tune threshold and top_k values.


In [ ]:

## Part 6: Real-World Application

### Understanding the Importance

# This final section bridges the gap between learning and application. You'll work with a more realistic scenario that includes challenges you'll face in production systems.

# **Why this matters:** Textbook examples rarely capture the messiness of real data. Learning to handle longer documents, mixed content types, and edge cases prepares you for actual RAG implementations.

# ### Task 6.1: Extended Knowledge Base

# Create a more complex dataset with longer documents.


EXTENDED_DOCS = [
    """
    Artificial Intelligence (AI) is transforming industries worldwide. Machine learning,
    a subset of AI, enables systems to learn from data without explicit programming.
    Companies use ML for predictive analytics, recommendation systems, and automation.
    The field has grown exponentially with the availability of big data and computational power.
    """,
    """
    Deep learning has revolutionized computer vision and natural language processing.
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving
    human-level performance in many cases. Applications include medical imaging,
    autonomous vehicles, and facial recognition systems.
    """,
    """
    Natural language processing enables machines to understand human language. Modern NLP
    uses transformer architectures like BERT and GPT. These models power chatbots,
    translation services, and search engines. NLP is crucial for human-computer interaction.
    """,

        """
    Computer vision focuses on enabling machines to interpret and analyze visual data.
    Techniques such as image classification, object detection, and segmentation
    allow systems to understand images and videos. It is widely used in healthcare,
    surveillance, robotics, and autonomous vehicles.
    """,

    """
    Reinforcement learning is a type of machine learning where an agent learns
    by interacting with its environment. The agent receives rewards or penalties
    based on its actions and improves its decision-making strategy over time.
    It is commonly applied in robotics, gaming, and recommendation systems.
    """,

    """
    Big data technologies support the storage and processing of massive datasets.
    Tools like distributed computing frameworks and cloud platforms enable scalable
    machine learning workflows. Efficient data pipelines are essential for training
    high-performance AI models.
    """
    # Add more documents as needed
]

print("Extended knowledge base created successfully.")
print("Total documents:", len(EXTENDED_DOCS))

Extended knowledge base created successfully.
Total documents: 6


In [ ]:


### Task 6.2: Handle Multi-Document Queries

# Implement a function that retrieves and combines information from multiple documents.

# ```python
def multi_document_retrieve(query, pipeline, top_k=5, combine=True):
    """
    Retrieve and optionally combine information from multiple documents.

    Args:
        query: Search query
        pipeline: RetrievalPipeline instance
        top_k: Number of documents to retrieve
        combine: Whether to combine retrieved documents

    Returns:
        Combined context string or list of documents
    """
    # Your code here
    # Retrieve top_k most relevant documents for the given query
    results = pipeline.retrieve(query, top_k=top_k)

    # Extract only the document text from (document, score) pairs
    retrieved_docs = [doc for doc, score in results]

    # Combine all retrieved documents into one single text
    # Each document is separated by two new lines
    combined_text = "\n\n".join(retrieved_docs)

    # Return the final combined text
    return combined_text



In [ ]:


### Task 6.3: Build a Complete RAG Retrieval Demo

# Create a simple interactive demo of your retrieval system.

# ```python
def interactive_retrieval_demo(pipeline):
    """
    Run an interactive demo of the retrieval system.
    """
    print("RAG Retrieval System Demo")
    print("=" * 50)
    print("Enter your queries (type 'quit' to exit)")
    print("=" * 50)

    # Your code here
    # Allow users to:
    # 1. Enter queries
    # 2. See retrieved documents with scores
    # 3. View statistics
        while True:
        # 1. Take user input
        query = input("\nEnter query: ")

        # Exit condition
        if query.lower() == "quit":
            print("Exiting demo...")
            break

        # 2. Retrieve top 3 relevant documents
        results = pipeline.retrieve(query, top_k=3)

        print("\nTop Retrieved Documents:\n")

        # Display rank, score and document text
        for i, (doc, score) in enumerate(results, 1):
            print("Rank:", i)
            print("Score:", round(score, 4))
            print("Document:", doc.strip())
            print("-" * 50)

        # 3. Display statistics
        print("Total documents retrieved:", len(results))



In [ ]:

## Challenge Tasks

### Challenge 1: Implement Re-ranking
#
# After initial retrieval, re-rank results using a different strategy (e.g., keyword matching combined with semantic similarity).
def rerank_results(query, pipeline, top_k=5):
    results = pipeline.retrieve(query, top_k=top_k)
    reranked = []
    query_words = query.lower().split()

    for doc, semantic_score in results:
        doc_words = doc.lower().split()
        keyword_score = 0

        for word in query_words:
            if word in doc_words:
                keyword_score += 1

        final_score = (semantic_score + keyword_score) / 2
        reranked.append((doc, final_score))

    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked
### Challenge 2: Add Metadata Filtering

# Extend your pipeline to support filtering documents by metadata (e.g., date, category, author).
DOCUMENTS_WITH_METADATA = [
    {"text": "AI is used in healthcare.", "category": "AI", "author": "John"},
    {"text": "Deep learning improves vision systems.", "category": "ML", "author": "Mary"},
    {"text": "NLP is used in chatbots.", "category": "AI", "author": "John"},
]

def filter_by_metadata(documents, category=None, author=None):
    filtered = []

    for doc in documents:
        if category and doc["category"] != category:
            continue
        if author and doc["author"] != author:
            continue
        filtered.append(doc)

    return filtered

### Challenge 3: Implement Hybrid Search

# Combine dense retrieval (embeddings) with sparse retrieval (keyword-based like BM25) for improved results.

def hybrid_search(query, pipeline, top_k=5):
    dense_results = pipeline.retrieve(query, top_k=top_k)
    hybrid_results = []
    query_words = query.lower().split()

    for doc, dense_score in dense_results:
        sparse_score = 0
        doc_words = doc.lower().split()

        for word in query_words:
            if word in doc_words:
                sparse_score += 1

        final_score = dense_score + sparse_score
        hybrid_results.append((doc, final_score))

    hybrid_results.sort(key=lambda x: x[1], reverse=True)
    return hybrid_results


In [ ]:
# ## Questions

# 1. What are the key trade-offs between chunk size and retrieval quality?
print("1. Small chunk size improves precision but may lose context.")
print("Large chunk size preserves context but may reduce precision.")
print("So there is a trade-off between accuracy and completeness of information.")
# 2. How would you handle documents in multiple languages?

print("\n2. Use multilingual embedding models.")
print("Detect language before indexing documents.")
print("Store language as metadata.")
print("Translate query if needed to match document language.")

# 3. What strategies could you use to update your knowledge base without recomputing all embeddings?

print("\n3. Compute embeddings only for new documents.")
print("Use incremental indexing.")
print("Update embeddings only for modified documents.")
print("Store embeddings in a vector database for easy updates.")
# 4. How would you monitor retrieval quality in a production system?
print("\n4. Track precision and recall metrics.")
print("Collect user feedback.")
print("Monitor click-through rates.")
print("Analyze failed or low-quality queries.")
# 5. What are the computational costs of your pipeline, and how would you optimize for large-scale deployment?

print("\n5. Costs include embedding computation, vector search, and storage.")
print("Optimize using vector databases like FAISS.")
print("Use batching, caching, and distributed systems.")
print("Reduce repeated computations where possible.")

1. Small chunk size improves precision but may lose context.
Large chunk size preserves context but may reduce precision.
So there is a trade-off between accuracy and completeness of information.

2. Use multilingual embedding models.
Detect language before indexing documents.
Store language as metadata.
Translate query if needed to match document language.

3. Compute embeddings only for new documents.
Use incremental indexing.
Update embeddings only for modified documents.
Store embeddings in a vector database for easy updates.

4. Track precision and recall metrics.
Collect user feedback.
Monitor click-through rates.
Analyze failed or low-quality queries.

5. Costs include embedding computation, vector search, and storage.
Optimize using vector databases like FAISS.
Use batching, caching, and distributed systems.
Reduce repeated computations where possible.
